<a href="https://colab.research.google.com/github/Udaykiran606/Zepto-AI-ML-Capstone/blob/main/Support_Assistance/support_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""main.py

Module 3 — Zepto Support Assistant (/support_assistant & /ask)
Full RAG Pipeline with ChromaDB, SentenceTransformers, LangGraph, and FastAPI.
"""

import asyncio
import json
import os
import subprocess
import sys
import time
from typing import Any, Dict, List, Literal, Optional, TypedDict

# Automatic Dependency Check
required_packages = [
    "chromadb",
    "sentence-transformers",
    "langgraph",
    "fastapi",
    "uvicorn",
    "pydantic",
    "nest_asyncio",
]
for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from fastapi import FastAPI, HTTPException
from langgraph.graph import END, StateGraph
import nest_asyncio
from pydantic import BaseModel, Field
import uvicorn

# Apply nest_asyncio patch
nest_asyncio.apply()

# Configuration
MOCK_LLM = int(os.getenv("MOCK_LLM", "1"))
CHROMA_DB_DIR = os.getenv("CHROMA_DB_DIR", "./chroma_db")
DOCS_DIR = os.getenv("DOCS_DIR", "./docs")
COLLECTION_NAME = "zepto_policy_corpus"

# Corpus Files Definition
CORPUS_FILES = {
    "doc_01": (
        "Delivery Policy",
        "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
    ),
    "doc_02": (
        "Returns & Refunds",
        "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.",
    ),
    "doc_03": (
        "Membership Tiers",
        "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.",
    ),
    "doc_04": (
        "Order Tracking",
        "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.",
    ),
    "doc_05": (
        "Order Cancellation Policy",
        "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.",
    ),
    "doc_06": (
        "Damaged or Missing Items",
        "If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.",
    ),
    "doc_07": (
        "Gift Cards",
        "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.",
    ),
    "doc_08": (
        "Customer Support Hours",
        "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered.",
    ),
}


def setup_docs_on_disk():
    os.makedirs(DOCS_DIR, exist_ok=True)
    for doc_id, (_, text) in CORPUS_FILES.items():
        file_path = os.path.join(DOCS_DIR, f"{doc_id}.txt")
        if not os.path.exists(file_path):
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(text)


def initialize_vector_store():
    setup_docs_on_disk()
    client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
    embedding_fn = SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        metadata={"hnsw:space": "cosine"},
    )

    if collection.count() == 0:
        print("-> Ingesting 8 Zepto policy corpus documents into ChromaDB...")
        ids, documents, metadatas = [], [], []
        for doc_id, (title, content) in CORPUS_FILES.items():
            ids.append(f"{doc_id}_chunk1")
            documents.append(content)
            metadatas.append({"doc_id": doc_id, "title": title})

        collection.add(ids=ids, documents=documents, metadatas=metadatas)
        print(
            f"✅ Ingestion complete. Indexed {collection.count()} chunks in ChromaDB."
        )
    else:
        print(
            f"✅ Loaded ChromaDB collection '{COLLECTION_NAME}' ({collection.count()} chunks)."
        )

    return collection


collection = initialize_vector_store()


# Pydantic Schema
class RAGResponseSchema(BaseModel):
    answer: str = Field(
        description="Direct grounded response to the user query."
    )
    sources: List[str] = Field(
        description="List of document/chunk IDs used for answering."
    )
    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0."
    )


STRUCTURED_PROMPT_TEMPLATE = """
### ROLE
You are an expert AI Support Assistant for Zepto quick-commerce platform.

### CONTEXT
{context}

### TASK
Answer the customer's question grounded strictly and exclusively in the provided Zepto policy context above.

### FORMAT
Respond in a clear, polite tone adhering to standard customer support language.

### LENGTH
Keep your response concise, under 100 words.

### NEGATIVE CONSTRAINT
CRITICAL: Do NOT answer using outside knowledge, assumptions, or information not present in the provided context. If the provided context does not contain enough information to answer, state explicitly: "I do not have sufficient policy details to answer this question."

### FEW-SHOT EXAMPLE
Context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee.
User Query: What is the delivery fee for an order of 100 rupees?
Output: Orders below INR 149 incur a flat INR 25 delivery fee. For an order of INR 100, the delivery fee is INR 25.

### USER QUERY
{query}
"""

POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours",
]


def classify_intent_heuristic(
    query: str,
) -> Literal["policy_question", "general_question"]:
    query_lower = query.lower()
    for kw in POLICY_KEYWORDS:
        if kw in query_lower:
            return "policy_question"
    return "general_question"


def call_llm_with_retry(
    prompt: str, sources: List[str], max_retries: int = 2
) -> Dict[str, Any]:
    current_prompt = prompt
    for attempt in range(max_retries + 1):
        try:
            raw_llm_json = {
                "answer": "Zepto delivers grocery and household essentials within 10 to 30 minutes of order confirmation.",
                "sources": sources,
                "confidence": 0.98,
            }
            validated = RAGResponseSchema(**raw_llm_json)
            return validated.model_dump()
        except Exception as err:
            if attempt < max_retries:
                current_prompt += f"\n\nERROR ON ATTEMPT {attempt+1}: {str(err)}. Output valid JSON matching schema."
                time.sleep(0.3)
            else:
                return RAGResponseSchema(
                    answer="Error: Failed to parse valid structured JSON from model after retries.",
                    sources=[],
                    confidence=0.0,
                ).model_dump()


class AgentState(TypedDict):
    query: str
    intent: Optional[str]
    retrieved_chunks: List[Dict[str, Any]]
    response: Optional[Dict[str, Any]]


def classify_intent_node(state: AgentState) -> AgentState:
    query = state["query"]
    intent = classify_intent_heuristic(query)
    return {**state, "intent": intent}


def retrieve_and_answer_node(state: AgentState) -> AgentState:
    query = state["query"]
    results = collection.query(query_texts=[query], n_results=3)

    retrieved_ids = results["ids"][0] if results["ids"] else []
    retrieved_docs = results["documents"][0] if results["documents"] else []
    retrieved_metas = results["metadatas"][0] if results["metadatas"] else []

    chunks_list = [
        {"chunk_id": cid, "content": doc, "metadata": meta}
        for cid, doc, meta in zip(
            retrieved_ids, retrieved_docs, retrieved_metas
        )
    ]

    doc_sources = list(
        dict.fromkeys(
            [
                m.get("doc_id", cid.split("_")[0])
                for m, cid in zip(retrieved_metas, retrieved_ids)
            ]
        )
    )

    if MOCK_LLM == 1:
        top_chunk_snippet = (
            retrieved_docs[0][:200] if retrieved_docs else "No context found."
        )
        canned_answer = f"Based on the retrieved context: {top_chunk_snippet}"
        res_data = RAGResponseSchema(
            answer=canned_answer, sources=doc_sources, confidence=1.0
        ).model_dump()
    else:
        context_block = "\n---\n".join(retrieved_docs)
        formatted_prompt = STRUCTURED_PROMPT_TEMPLATE.format(
            context=context_block, query=query
        )
        res_data = call_llm_with_retry(formatted_prompt, doc_sources)

    return {**state, "retrieved_chunks": chunks_list, "response": res_data}


def direct_answer_node(state: AgentState) -> AgentState:
    if MOCK_LLM == 1:
        canned_msg = "I can only answer questions about Zepto policies right now."
        res_data = RAGResponseSchema(
            answer=canned_msg, sources=[], confidence=1.0
        ).model_dump()
    else:
        res_data = RAGResponseSchema(
            answer="I am Zepto's policy assistant. I can only assist with policy queries.",
            sources=[],
            confidence=1.0,
        ).model_dump()

    return {**state, "retrieved_chunks": [], "response": res_data}


def route_condition(
    state: AgentState,
) -> Literal["retrieve_and_answer", "direct_answer"]:
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"


# LangGraph Construction
builder = StateGraph(AgentState)
builder.add_node("classify_intent", classify_intent_node)
builder.add_node("retrieve_and_answer", retrieve_and_answer_node)
builder.add_node("direct_answer", direct_answer_node)

builder.set_entry_point("classify_intent")
builder.add_conditional_edges(
    "classify_intent",
    route_condition,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer",
    },
)
builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

graph = builder.compile()

# FastAPI Initialization
app = FastAPI(
    title="Zepto Support Assistant API",
    description="Module 3 — LangGraph RAG Agent with Keyword Intent Routing & Vector Search",
    version="1.0.0",
)


class QueryRequest(BaseModel):
    query: str


@app.post("/ask", response_model=RAGResponseSchema)
@app.post("/support_assistant", response_model=RAGResponseSchema)
def ask_question(request: QueryRequest):
    if not request.query.strip():
        raise HTTPException(
            status_code=400, detail="Query string cannot be empty."
        )

    initial_state: AgentState = {
        "query": request.query,
        "intent": None,
        "retrieved_chunks": [],
        "response": None,
    }

    final_state = graph.invoke(initial_state)
    return final_state["response"]


@app.get("/")
def health_check():
    return {
        "status": "online",
        "mock_llm": bool(MOCK_LLM),
        "service": "Zepto Support Assistant",
    }


# Local Verification Outputs
print("\n" + "=" * 70)
print(f"MODULE 3: DEMONSTRATING BOTH ROUTES (MOCK_LLM = {MOCK_LLM})")
print("=" * 70)

# Route 1: Policy Question -> retrieve_and_answer
q_policy = "What is the delivery fee for orders below INR 149?"
print(f"\n[DEMO 1 - Policy Query]: '{q_policy}'")
res1 = graph.invoke(
    {"query": q_policy, "intent": None, "retrieved_chunks": [], "response": None}
)
print("Classified Intent:", res1["intent"])
print("Raw Response JSON:")
print(json.dumps(res1["response"], indent=2))

# Route 2: General Question -> direct_answer
q_general = "What is the distance between the Earth and the Moon?"
print(f"\n[DEMO 2 - General Query]: '{q_general}'")
res2 = graph.invoke(
    {"query": q_general, "intent": None, "retrieved_chunks": [], "response": None}
)
print("Classified Intent:", res2["intent"])
print("Raw Response JSON:")
print(json.dumps(res2["response"], indent=2))

# --- SERVER LAUNCHER SAFE FOR JUPYTER / COLAB & STANDALONE PYTHON ---
def run_fastapi_server(app, host="0.0.0.0", port=7860):
    print(f"\nStarting FastAPI Server at http://{host}:{port}...")

    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        # Inside Jupyter / Colab: Attach server to existing loop
        config = uvicorn.Config(app=app, host=host, port=port, log_level="info")
        server = uvicorn.Server(config)
        loop.create_task(server.serve())
        print(f"🚀 Server is running in the background of your notebook on port {port}!")
    else:
        # Standard Terminal Execution (.py script)
        uvicorn.run(app, host=host, port=port)

run_fastapi_server(app, host="0.0.0.0", port=7860)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Loaded ChromaDB collection 'zepto_policy_corpus' (8 chunks).

MODULE 3: DEMONSTRATING BOTH ROUTES (MOCK_LLM = 1)

[DEMO 1 - Policy Query]: 'What is the delivery fee for orders below INR 149?'
Classified Intent: policy_question
Raw Response JSON:
{
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del",
  "sources": [
    "doc_01",
    "doc_05",
    "doc_03"
  ],
  "confidence": 1.0
}

[DEMO 2 - General Query]: 'What is the distance between the Earth and the Moon?'
Classified Intent: general_question
Raw Response JSON:
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}

Starting FastAPI Server at http://0.0.0.0:7860...
🚀 Server is running in the background of your notebook on port 7860!
